# NB-04 · Augmentation & Dataset Pipeline Diagnostics

**Checks covered**
1. Augmentation applied only during `train` split, not `val`/`test`
2. After flip/rotate: input and GT stay spatially aligned
3. Noise augmentation: signal-to-noise ratio stays reasonable
4. DataLoader: no NaN/Inf in batches, correct shapes
5. Train / val / test split regions: no overlap, correct patch counts
6. Reproducibility: same index returns identical result when augmentation is off

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

REPO_ROOT    = Path("../..").resolve()
DATASET_PATH = Path("/ste/rnd/User/vice_vi/Dataset/clean_dataset")
PARAMS_PATH  = DATASET_PATH / "params" / "params_sig_k5" / "parameters_sig_k5.npy"
DATA_DIR     = DATASET_PATH / "data"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tools.logger                           import Logger
from tools.crop_region                      import CropRegion
from configuration.dataset_config           import InputConfig, OutputConfig, AugmentationConfig, PatchConfiguration, Representation
from pipelines.dataset_pipeline.patch       import Patcher
from pipelines.dataset_pipeline.load        import PatchDataset
from pipelines.dataset_pipeline.augmentation import SpatialAugmenter
from pipelines.dataset_pipeline.normalize   import Stats, StatsComputer, Normalizer

logger = Logger(name="nb04", logdir="/tmp", run_name="nb04", verbose=False)

with open(DATA_DIR / "dataset.json") as f:
    layout = json.load(f)

gc = CropRegion(*layout["global_crop"])

PATCH_SIZE  = (64, 64)
STRIDE      = 32
N_GAUSSIANS = 5

SPLITS = {
    "train" : (1000,  9120),
    "val"   : (9120,  12400),
    "test"  : (12400, 16000),
}

primary        = np.load(next(DATA_DIR.glob("primary_*.npy")),        mmap_mode='r')
secondaries    = np.load(next(DATA_DIR.glob("secondaries_*.npy")),    mmap_mode='r')
interferograms = np.load(next(DATA_DIR.glob("interferograms_*.npy")), mmap_mode='r')
params_full    = np.load(PARAMS_PATH,                                 mmap_mode='r')

input_config  = InputConfig(
    use_primary=True,  primary_representation=Representation.MAG_ONLY,
    use_secondaries=True, secondaries_representation=Representation.MAG_ONLY,
    use_interferograms=True, interferograms_representation=Representation.ANGLE_ONLY,
)
output_config = OutputConfig()

print("Setup complete")

## 1 · Split regions: no overlap, correct patch counts

In [ ]:
split_ends = list(SPLITS.values())
print(f"  {'Split':<8s}  {'az_start':>8s}  {'az_end':>8s}  {'H':>6s}  {'W':>6s}  {'Patches':>8s}")
print("  " + "-" * 55)
for name, (a0, a1) in SPLITS.items():
    H, W     = a1 - a0, gc.range_end - gc.range_start
    p        = Patcher.build((H, W), PATCH_SIZE, STRIDE)
    n_patches= p.grid.number_of_patches
    print(f"  {name:<8s}  {a0:>8d}  {a1:>8d}  {H:>6d}  {W:>6d}  {n_patches:>8d}")

# Check no overlap
pairs = [("train","val"), ("val","test"), ("train","test")]
print()
for a_name, b_name in pairs:
    a_end   = SPLITS[a_name][1]
    b_start = SPLITS[b_name][0]
    overlap = max(0, a_end - b_start)
    status  = "✓" if overlap == 0 else f"⚠ OVERLAP = {overlap}"
    print(f"  {a_name}-{b_name} overlap: {status}")

## 2 · Augmentation: only during train split

In [ ]:
aug_cfg   = AugmentationConfig(p_flip_h=0.5, p_flip_v=0.5, p_rot90=0.5, p_noise=0.3, noise_std=0.1)
augmenter = SpatialAugmenter(aug_cfg, logger)

az0, az1 = SPLITS["train"]
rg0, rg1 = gc.range_start, gc.range_end

inputs_tr = np.concatenate([
    primary[np.newaxis, az0:az1, rg0:rg1],
    secondaries[:, az0:az1, rg0:rg1],
    interferograms[:, az0:az1, rg0:rg1],
], axis=0).astype(np.complex64)
params_tr = params_full[:, az0:az1, rg0:rg1].astype(np.float32)
patcher_tr = Patcher.build((inputs_tr.shape[1], inputs_tr.shape[2]), PATCH_SIZE, STRIDE)

# Collect augmented variants of the same patch
ds_train = PatchDataset(
    inputs=inputs_tr, gt_parameters=params_tr, grid=patcher_tr,
    input_config=input_config, output_config=output_config,
    split_name="train", n_gaussians=N_GAUSSIANS, augmenter=augmenter,
)
ds_val = PatchDataset(
    inputs=inputs_tr, gt_parameters=params_tr, grid=patcher_tr,
    input_config=input_config, output_config=output_config,
    split_name="val",   n_gaussians=N_GAUSSIANS, augmenter=augmenter,  # augmenter present but split=val
)

np.random.seed(0)
x_val_a, _ = ds_val[0]
np.random.seed(0)
x_val_b, _ = ds_val[0]
val_identical = np.allclose(x_val_a, x_val_b)
print(f"Val split: same patch sampled twice is identical? {val_identical}  {'✓' if val_identical else '⚠ BUG: augmentation active in val'}")

## 3 · Spatial alignment after augmentation

In [ ]:
# For each flip/rotation: check that a landmark in the input moves the same way in GT
rng = np.random.default_rng(1)

# Create a deterministic test: place a bright pixel at a known location
_, y_raw = ds_train.grid.extract(params_tr, 100), None
x_raw_patch, y_raw_patch = PatchDataset(
    inputs=inputs_tr, gt_parameters=params_tr, grid=patcher_tr,
    input_config=input_config, output_config=output_config,
    split_name="train", n_gaussians=N_GAUSSIANS, augmenter=None,
)[100]

test_x   = x_raw_patch.copy()
test_y   = y_raw_patch.copy()
ph, pw   = PATCH_SIZE

# Manually apply each augmentation and verify GT moves in sync
def check_alignment(x, y, label):
    # Find max in input ch0 and GT ch0; they should be at the same spatial location
    pos_x = np.unravel_index(np.argmax(np.abs(x[0])), x[0].shape)
    pos_y = np.unravel_index(np.argmax(np.abs(y[0])), y[0].shape)
    aligned = pos_x == pos_y
    print(f"  {label:<25s}: input max @ {pos_x}, GT max @ {pos_y}  {'✓' if aligned else '⚠ MISALIGNED'}")

# Set a known spike in a patch
spike = test_x.copy(); spike[0, 20, 30] = 1e6
spike_y = test_y.copy(); spike_y[0, 20, 30] = 1e6

# flip H
fx = np.flip(spike,   axis=-1).copy()
fy = np.flip(spike_y, axis=-1).copy()
check_alignment(fx, fy, "flip horizontal")

# flip V
fx = np.flip(spike,   axis=-2).copy()
fy = np.flip(spike_y, axis=-2).copy()
check_alignment(fx, fy, "flip vertical")

# rot90
for k in [1, 2, 3]:
    rx = np.rot90(spike,   k=k, axes=(-2,-1)).copy()
    ry = np.rot90(spike_y, k=k, axes=(-2,-1)).copy()
    check_alignment(rx, ry, f"rot90 k={k}")

## 4 · Noise augmentation: SNR check

In [ ]:
# The noise augmenter should not destroy the signal
aug_noise = SpatialAugmenter(
    AugmentationConfig(p_flip_h=0.0, p_flip_v=0.0, p_rot90=0.0, p_noise=1.0, noise_std=0.1),
    logger
)

x_ref = x_raw_patch.copy()
x_noisy, _ = aug_noise(x_ref.copy(), y_raw_patch.copy())

noise   = x_noisy - x_ref
sig_pwr = float(np.mean(x_ref**2))
nse_pwr = float(np.mean(noise**2))
snr_db  = 10 * np.log10(sig_pwr / (nse_pwr + 1e-12))

print(f"Signal power   : {sig_pwr:.4f}")
print(f"Noise power    : {nse_pwr:.4f}")
print(f"SNR            : {snr_db:.1f} dB")
if snr_db < 5:
    print("⚠  Very low SNR — noise_std may be too high relative to the signal")
else:
    print("✓  Noise level appears reasonable")

## 5 · DataLoader: NaN/Inf scan across batches

In [ ]:
loader = DataLoader(ds_train, batch_size=64, shuffle=True, num_workers=0, drop_last=False)

N_BATCHES  = 20
nan_batches, inf_batches = [], []
shape_mismatches         = []

expected_x_shape = None
expected_y_shape = None

for b_idx, (x_batch, y_batch) in enumerate(loader):
    if b_idx >= N_BATCHES:
        break

    if expected_x_shape is None:
        expected_x_shape = x_batch.shape[1:]
        expected_y_shape = y_batch.shape[1:]
    elif x_batch.shape[1:] != expected_x_shape or y_batch.shape[1:] != expected_y_shape:
        shape_mismatches.append(b_idx)

    if torch.isnan(x_batch).any() or torch.isnan(y_batch).any():
        nan_batches.append(b_idx)
    if torch.isinf(x_batch).any() or torch.isinf(y_batch).any():
        inf_batches.append(b_idx)

print(f"Checked {min(N_BATCHES, b_idx+1)} batches")
print(f"Expected input  shape : {expected_x_shape}")
print(f"Expected output shape : {expected_y_shape}")
print(f"NaN batches    : {nan_batches  if nan_batches   else '✓ none'}")
print(f"Inf batches    : {inf_batches  if inf_batches   else '✓ none'}")
print(f"Shape mismatches: {shape_mismatches if shape_mismatches else '✓ none'}")

## 6 · Visual: augmented patch pairs

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(14, 12))
fig.suptitle("Augmented patch pairs (input ch0 | GT ch0)", fontsize=12)

for row in range(4):
    x_aug, y_aug = ds_train[row * 10]
    axes[row, 0].imshow(x_raw_patch[0], cmap="gray", aspect="auto"); axes[row, 0].set_title("Input raw")
    axes[row, 1].imshow(x_aug[0],       cmap="gray", aspect="auto"); axes[row, 1].set_title(f"Input aug (sample {row*10})")
    axes[row, 2].imshow(y_raw_patch[0], cmap="RdBu_r", aspect="auto"); axes[row, 2].set_title("GT raw")
    axes[row, 3].imshow(y_aug[0],       cmap="RdBu_r", aspect="auto"); axes[row, 3].set_title("GT aug")

for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()